# ML-05 — Feature Vector and Leakage/Privacy Check

This notebook defines the model input using fields available before the observed snapshot-proxy label. IDs are retained only for grouping and tracing; label-derived fields are excluded.

## 1. Build the feature vector

Numeric fields are median-imputed and receive missingness flags. Categorical context is one-hot encoded. The implementation is shared with the validation notebook so the feature contract cannot silently drift.

In [1]:
from pathlib import Path
import sys
import pandas as pd

repo_candidates = [Path.cwd(), *Path.cwd().parents, Path('/content/FlyRank-ML'), Path('/content/flyrank-ml')]
repo_root = next((p for p in repo_candidates if (p / 'work' / 'ml_track.py').exists()), Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from work.ml_track import (
    CATEGORICAL_FEATURES,
    NUMERIC_FEATURES,
    TARGET,
    ensure_dirs,
    load_analysis_frame,
    make_feature_matrix,
    run_artifacts,
    run_validation,
    write_json,
    write_paper_page,
)

ensure_dirs()
frame = load_analysis_frame()
print(f"Loaded {len(frame):,} rows across {frame['client_id'].nunique():,} client groups")
print(f"Observed snapshot-proxy base rate: {frame[TARGET].mean():.3f}")

features, feature_names = make_feature_matrix(frame)
print(f'Feature matrix shape: {features.shape}')
print(f'Feature names: {len(feature_names)}')

Warehouse unavailable; using starter slice (ImportError).
Loaded 30,000 rows across 32 client groups
Observed snapshot-proxy base rate: 0.542


Feature matrix shape: (30000, 79)
Feature names: 79


C:\Users\khali\AppData\Roaming\Python\Python313\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\khali\AppData\Roaming\Python\Python313\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\khali\AppData\Roaming\Python\Python313\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\khali\AppData\Roaming\Python\Python313\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\khali\AppData\Roaming\Python\Python313\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


## 2. Feature notes (meaning, missing, categorical, available-when?)

The performance fields describe the prior 30-day window, content age and update age describe the page before the snapshot, and categorical fields describe stable context such as device or search type. Missing numeric values are imputed with a training-fold median and marked with an indicator; missing categorical values become an explicit category. These fields are available before the prediction snapshot.

In [2]:
numeric_present = [name for name in NUMERIC_FEATURES if name in frame.columns]
categorical_present = [name for name in CATEGORICAL_FEATURES if name in frame.columns]
print('Numeric inputs:', numeric_present)
print('Categorical inputs:', categorical_present)
print('Missing-value policy: median + missingness flag for numeric; explicit category for categorical.')
print('Availability check: prior-window and page-history fields precede the observed label window.')

Numeric inputs: ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'users_prev_30d', 'engaged_sessions_prev_30d', 'ai_sessions_prev_30d', 'scroll_events_prev_30d', 'days_with_impressions_prev_30d', 'days_with_sessions_prev_30d', 'content_age_days', 'age_tier_order', 'days_since_last_update', 'ctr_prev_30d', 'avg_position_prev_30d', 'engagement_rate_prev_30d', 'scroll_rate_prev_30d', 'ai_traffic_pct_prev_30d', 'query_impressions_prev_30d', 'query_clicks_prev_30d', 'query_count_prev_30d', 'query_position_prev_30d', 'query_ctr_prev_30d']
Categorical inputs: ['competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier']
Missing-value policy: median + missingness flag for numeric; explicit category for categorical.
Availability check: prior-window and page-history fields precede the observed label window.


## 3. The leakage hunt

The target is derived from the trend fields. A valid feature matrix must contain neither the target nor its source columns, and must not use identifiers as predictive features. The deliberately leaky diagnostic is reported only to show why the trend field is forbidden.

In [3]:
forbidden = {'trend_pct', 'trend_direction', TARGET, 'content_id', 'client_id'}
found = sorted(forbidden.intersection(feature_names))
assert not found, f'Forbidden feature(s) found: {found}'
print('Forbidden feature columns present:', found)
print('Leakage verdict: PASS — label-derived fields and IDs are excluded.')

Forbidden feature columns present: []
Leakage verdict: PASS — label-derived fields and IDs are excluded.


## 4. What I excluded and why

- `trend_pct` and `trend_direction`: they define the snapshot-proxy label.
- `target`: direct label leakage.
- `content_id` and `client_id`: identifiers, not page behavior; `client_id` is used only for grouped validation.
- Any future-window or post-intervention field: unavailable at decision time.

In [4]:
excluded = {
    'trend_pct': 'source of the label',
    'trend_direction': 'source of the label',
    TARGET: 'direct label',
    'content_id': 'identifier',
    'client_id': 'grouping identifier, not a feature',
}
print('Excluded fields:', excluded)
print('Public-safety verdict: PASS — no client names, URLs, private queries, or credentials are used.')

Excluded fields: {'trend_pct': 'source of the label', 'trend_direction': 'source of the label', 'is_declining_label': 'direct label', 'content_id': 'identifier', 'client_id': 'grouping identifier, not a feature'}
Public-safety verdict: PASS — no client names, URLs, private queries, or credentials are used.


## Self-check

- [x] Every section contains both reasoning and executable checks
- [x] The notebook uses the shared feature contract
- [x] Label-derived fields and identifiers are excluded
- [x] Claims use observed, measured, directional, and decision-support language